# 00 — Project setup, data strategy, and NLP product framing

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## What this notebook gives you

Use this first notebook as the reusable skeleton for company or portfolio projects.

You will set up:

- a toy customer-text dataset
- train/validation/test splits
- reusable classification metrics
- a decision tree for choosing rules, classical ML, Transformers, RAG, or fine-tuning
- a simple model card template

**PM/founder framing:** before choosing a model, define the user job, output format, data availability, and cost of errors.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = DATA_DIR / "sample_customer_tickets.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## Basic data audit

Before training anything, check the distribution of labels. Many NLP failures are data failures: unclear labels, class imbalance, duplicates, or inconsistent annotation rules.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())
print("\nIntent distribution:")
print(df["intent"].value_counts())
print("\nUrgency distribution:")
print(df["urgency"].value_counts())

In [ ]:
# A tiny split for demonstration. For real projects, use more data and stratify carefully.
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["sentiment"])
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["sentiment"])

for name, split in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(name, split.shape)
    print(split["sentiment"].value_counts().to_dict())

## Reusable evaluation helpers

Use these helpers across classification tasks such as sentiment, intent, toxicity, urgency, routing, and churn-risk detection.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support


def evaluate_classification(y_true, y_pred, labels=None):
    """Return a compact dictionary plus print a report."""
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print("Confusion matrix labels:", labels)
    print(cm)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

# Dummy baseline: always predict most frequent class
majority = train_df["sentiment"].mode()[0]
preds = [majority] * len(test_df)
evaluate_classification(test_df["sentiment"], preds, labels=sorted(df["sentiment"].unique()))

## NLP system choice decision tree

Use this as a PM/founder shortcut:

| Situation | Usually start with |
|---|---|
| Simple deterministic extraction | Regex/rules |
| Cheap high-volume classification | TF-IDF + logistic regression/SVM |
| Token-level labels | BERT token classifier or CRF baseline |
| Generating drafts/summaries | LLM prompting or encoder-decoder model |
| Private/current knowledge needed | RAG |
| Consistent repeated task with labelled data | Fine-tuning or LoRA |
| High-stakes output | Retrieval, citations, validators, human review |

In [ ]:
def choose_nlp_approach(output_type, has_labels=False, needs_private_knowledge=False, high_stakes=False, needs_generation=False):
    """Simple heuristic decision helper. Adapt this to your company workflow."""
    if high_stakes and needs_private_knowledge:
        return "RAG + citations + validators + human review"
    if needs_private_knowledge:
        return "RAG before fine-tuning"
    if output_type == "token_labels":
        return "BERT token classification; use CRF/HMM as baseline"
    if needs_generation:
        return "Prompting for MVP; fine-tune/LoRA if behavior must be consistent"
    if output_type == "single_label" and has_labels:
        return "TF-IDF/logistic baseline, then BERT/fine-tune if needed"
    if output_type == "single_label":
        return "Prompting or distant supervision to bootstrap labels"
    return "Start with rules + manual error analysis"

examples = [
    dict(output_type="single_label", has_labels=True),
    dict(output_type="token_labels", has_labels=True),
    dict(output_type="answer", needs_private_knowledge=True, high_stakes=True, needs_generation=True),
]
for e in examples:
    print(e, "=>", choose_nlp_approach(**e))

## Model card starter template

A model card helps your portfolio and your company by documenting what the model is for, what data it saw, how it was evaluated, and when not to use it.

In [ ]:
model_card = {
    "model_name": "sentiment-baseline-v0",
    "task": "customer ticket sentiment classification",
    "intended_use": "route negative/high-risk customer messages for support review",
    "not_for": "making punitive decisions about customers or employees",
    "training_data": "sample_customer_tickets.csv; replace with production-labelled tickets",
    "metrics": {"macro_f1": None, "precision_negative": None, "recall_negative": None},
    "known_limitations": ["sarcasm", "mixed sentiment", "new slang", "domain shift"],
    "human_review_required_when": ["high urgency", "low model confidence", "legal/privacy complaint"],
}
model_card